# ANPR Узбекистан — демо

Распознавание автомобильных номеров: детекция пластин **YOLO11** + чтение текста
**TPS-ResNet-BiLSTM-CTC**. Распознаватель обучен автором проекта на собственном датасете.

- Код: https://github.com/IzzatulloOne/plate-recognition-uz
- Модель: https://huggingface.co/IzzatulloOne/anpr-uz

Форматы: частные `01 A 123 BC`, юрлица `01 123 ABC`, прицепы `4169 AA 70`
(регион в конце), электромобили `01 M 018461`, иностранные `01 H 018461`.

Измеренная точность — 72 из 80 номеров на выборке с platesmania.

**Как пользоваться:** «Среда выполнения» → «Выполнить все», затем в предпоследней
ячейке загрузить фото. GPU не нужен, всё считается на CPU.

## 1. Установка

Ставим только то, чего нет в Colab: torch, numpy и opencv там уже есть.
Пакет проекта ставится без зависимостей, чтобы не тянуть за собой другую версию torch.

In [ ]:
import os

REPO = 'https://github.com/IzzatulloOne/plate-recognition-uz.git'
# Папка переживает перезапуск среды, поэтому clone со второго раза падает.
# Веса лежат внутри неё — удалять нельзя, обновляем на месте.
if os.path.exists('/content/anpr/.git'):
    !cd /content/anpr && git fetch -q --depth 1 origin main && git reset -q --hard origin/main
else:
    !git clone -q --depth 1 $REPO /content/anpr

%cd /content/anpr
!pip install -q ultralytics pydantic-settings huggingface_hub
!pip install -q --no-deps -e .
print('готово')


## 2. Веса

Детектор скачивается скриптом проекта, распознаватель (188 МБ) — с HF Hub.

In [ ]:
%cd /content/anpr
from huggingface_hub import hf_hub_download

!python -m tools.fetch_yolo

hf_hub_download(
    repo_id='IzzatulloOne/anpr-uz',
    filename='best_accuracy.pth',
    local_dir='/content/anpr',
)
print('веса на месте')

## 3. Загрузка моделей

Первый прогон дороже остальных — выделяется память и инициализируются ядра,
поэтому сразу делаем холостой кадр.

In [ ]:
%cd /content/anpr
import time

import numpy as np

from app.config import Settings
from app.core.pipeline import ANPRPipeline

settings = Settings(save_snapshots=False)
t0 = time.time()
pipeline = ANPRPipeline(settings)
pipeline.process_frame(np.zeros((720, 1280, 3), np.uint8), allow_retry=False)
print(f'модели загружены и прогреты за {time.time() - t0:.0f} с')

## 4. Своё фото

Запустите ячейку и выберите один или несколько файлов.

In [ ]:
import cv2
import numpy as np
from google.colab import files
from google.colab.patches import cv2_imshow

from app.core import plate_rules

# После перезапуска среды переменные обнуляются, и pipeline исчезает.
# Лучше сказать это прямо, чем уронить NameError где-то внутри цикла.
if "pipeline" not in globals():
    raise RuntimeError(
        "Модели не загружены. Выполните ячейку «Загрузка моделей» выше, "
        "либо «Среда выполнения» → «Выполнить всё»."
    )

uploaded = files.upload()

for name, blob in uploaded.items():
    frame = cv2.imdecode(np.frombuffer(blob, np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        print(f'{name}: не изображение')
        continue

    res = pipeline.process_frame(frame)
    print(f'\n=== {name} ===')
    print(f'детекция {res.detect_ms:.0f} мс, распознавание {res.recognize_ms:.0f} мс')

    if not res.plates:
        print('пластин не найдено')

    for p in res.plates:
        m = plate_rules.match(p.text)
        mark = 'OK ' if p.valid else '?  '
        print(f'  {mark} {m.pretty():16} conf={p.conf:.2f}  формат={p.format or "не опознан"}')
        x1, y1, x2, y2 = p.box
        color = (0, 200, 0) if p.valid else (0, 165, 255)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, p.text, (x1, max(20, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    scale = min(1.0, 1100 / frame.shape[1])
    cv2_imshow(cv2.resize(frame, None, fx=scale, fy=scale) if scale < 1 else frame)

## 5. Что стоит знать про результат

- **Пластины уже 70 пикселей на видео отбрасываются намеренно.** На таком размере
  модель выдаёт уверенно неверный ответ: в замере кроп 58×28 читался с уверенностью
  0.965 и валидным форматом, хотя правильный текст не смог разобрать и человек.
  Промолчать честнее, чем выдать правдоподобную ошибку. В этом блокноте порог
  меньше — здесь одиночное фото, а не поток.
- **Двухстрочные квадратные номера** (прицепы, спецтехника) текущий чекпоинт не читает.
- **Цвет фона определяется эвристикой по HSV** и заметно врёт на тенях.
- На видео поверх этого работают трекинг и голосование по кадрам: там точность выше,
  чем на одиночном снимке.